## 🚀 Creating Your First Pipeline Using the Copy Activity (Step-by-Step)

### Step 1: Open the Authoring Hub
1. In the Azure Data Factory Studio, click on the **Author (✏️)** icon in the left-hand navigation menu to open your workspace.

### Step 2: Create a New Pipeline
1. Hover over the **Pipelines** header in the factory resources pane, click the **`...` (Actions)** button, and select **New pipeline**.
2. In the properties pane on the right, give your pipeline a clear name (e.g., `PL_Copy_SqlToDataLake`).

### Step 3: Add the Copy Activity to the Canvas
1. In the **Activities** panel on the left, expand the **Move and transform** category.
2. Drag and drop the **Copy data** activity onto the main visual pipeline canvas.
3. Select the activity on the canvas and update its name in the **General** tab if desired (e.g., `Copy_SQL_To_ADLS`).

### Step 4: Configure the Source Tab
1. Click on the **Source** tab at the bottom of the canvas properties window.
2. Click **+ New** to create a new source dataset.
3. Choose your source data store type (e.g., **Azure SQL Database**) and click *Continue*.
4. Select an existing **Linked Service** (or click *New* to configure connection credentials for your source database).
5. Choose the specific source table from the dropdown menu and click *OK*.

### Step 5: Configure the Sink (Destination) Tab
1. Click on the **Sink** tab next to the Source tab.
2. Click **+ New** to create a new sink dataset.
3. Choose your destination data store type (e.g., **Azure Data Lake Storage Gen2**) and click *Continue*.
4. Select the file format (e.g., *DelimitedText* or *Parquet*).
5. Link it to your destination Linked Service and browse to target container/folder path, then click *OK*.

### Step 6: Validate and Run / Debug
1. **Validate**: Click the **Validate** button at the top of the pipeline canvas to ensure there are no configuration errors or missing properties.
2. **Debug**: Click the **Debug** button to execute an immediate test run of your pipeline.
3. **Monitor**: Switch over to the **Monitor (📊)** tab on the left navigation sidebar to view your pipeline run status, check row counts copied, and review success logs.

## 📝 Ingesting a CSV File from GitHub using ADF Copy Activity

#### 1. Create the Pipeline and Copy Activity
1. **Create Pipeline**: Went to the **Author (✏️)** hub, clicked on **Pipelines**, and created a new pipeline.
2. **Add Copy Activity**: Expanded **Move and transform** in the activities panel and dragged the **Copy data** activity onto the canvas.

#### 2. Source Setup (GitHub via HTTP Connector)
To read a raw file directly from GitHub without cloning the repo, you use the **HTTP** connector in ADF:
* **Linked Service Connection**: Created a Linked Service pointing to the base URL of GitHub.
  * **Base URL added in Source / Linked Service**: `https://raw.githubusercontent.com/`
* **Dataset / Relative URL Path**: Pointed to the exact repository path and branch to locate the target file.
  * **Relative URL / File Path added**: `Apoorvasmr/My_azure_notes/refs/heads/main/Fact_Sales_2.csv`
* **File Format**: Configured as **DelimitedText** (CSV), ensuring ADF correctly recognized the comma delimiter and header row.

#### 3. Sink (Destination) Setup
* Configured the target destination (e.g., an Azure Data Lake Storage Gen2 container or an Azure SQL table) where the raw sales data should be written.
* Linked it to the corresponding destination Linked Service and folder/table path.

#### 4. Debugging and Testing
* Clicked the **Debug** button at the top of the pipeline canvas to execute an immediate test run of the copy job.
* Switched to the **Monitor (📊)** tab to verify that the pipeline successfully connected to GitHub, downloaded `Fact_Sales_2.csv`, and successfully loaded the records into the Azure destination.


## ⚙️ How to Configure Get Metadata in ADF
1. **Add Activity**: In the Author tab, expand the **General** category in the activities panel and drag **Get metadata** onto your pipeline canvas.
2. **Dataset Tab**: 
   * Select an existing dataset (or create a new one pointing to your file or table) that you want to inspect.
3. **Field List Tab**: 
   * Click **+ New** to select the specific metadata properties you want to extract. Common options include:
    * **Child items**: Returns a list of all files and subfolders located inside a specified container or directory.
    * **Exists**: Returns a boolean (`true` or `false`) indicating whether the specified file, table, or folder actually exists.
    * **Item name**: Returns the name of the file or folder being inspected.
    * **Item type**: Specifies whether the inspected item is a File or a Folder (Container).
    * **Last modified**: Returns a timestamp showing the exact date and time the file or folder was last updated.
    * **Add dynamic content**: An expression builder tool used to write code referencing outputs and variables from previous activities.
4. **Using the Output**: 
   * You can reference the output of a Get Metadata activity in subsequent activities using dynamic content expressions (e.g., `@activity('GetMetadata1').output.exists` or `@activity('GetMetadata1').output.childItems`).

## ⚙️ How to Configure If Condition in ADF
1. **Add Activity**: In the Author tab, expand the **Iteration & conditionals** category in the activities panel and drag **If Condition** onto your pipeline canvas.
2. **Settings Tab (The Expression)**: 
   * Click into the **Expression** box and click **Add dynamic content** to write the logical condition (e.g., `@activity('GetMetadata1').output.exists` or `@equals(activity('GetMetadata1').output.itemType, 'Folder')`).
3. **Activities Tab (True / False Blocks)**: 
   * Inside the If Condition activity properties, click on the pencil icon for the **True** activities block to drop in the steps that should run if the expression evaluates to true.
   * Do the same for the **False** activities block to build out the alternative execution path.

## ⚙️ How to Configure ForEach in ADF
1. **Add Activity**: In the Author tab, expand the **Iteration & conditionals** category in the activities panel and drag **For each** onto your pipeline canvas.
2. **Settings Tab**:
   * **Batch count**: Controls how many items are processed simultaneously (leave unchecked for default sequential processing, or set a number for parallel execution up to 50).
   * **Items**: Click into the box, open **Add dynamic content**, and provide the array or collection you want to loop through (e.g., pointing to the output of a Get Metadata activity: `@activity('GetMetadata1').output.childItems`).
3. **Activities Tab (Inside the Loop)**:
   * Click on the pencil icon inside the ForEach activity box to open its internal canvas.
   * Drag and drop the activities you want repeated for each item (such as a **Copy Activity**).
   * Inside those nested activities, use the loop item variable—`@item().name`—to dynamically reference the current file or table being processed in that iteration.

## 🎯 Hands-On Lab: Filtering and Copying Specific Files Using ADF Control Flow

### Overview of the Scenario
When a container or folder contains multiple files, but you only want to extract a specific subset (e.g., files starting with `"Fact"`), you combine **Get Metadata**, **ForEach**, **If Condition**, and a **Parameterized Copy Activity** to automate the selective migration.

### Step-by-Step Implementation Flow

#### 1. Step 1: Inspect Folder Contents (Get Metadata Activity)
* **Pipeline Setup**: Created a new pipeline and dragged the **Get Metadata** activity onto the canvas.
* **Dataset Configuration**: Pointed it to the source container/folder.
* **Field List**: Selected **Child items** to retrieve a complete array/list of all files and folders present in the directory.

#### 2. Step 2: Loop Through All Files (ForEach Activity)
* **Add Activity**: Dragged the **For each** activity and connected it to the Get Metadata activity.
* **Settings Configuration**:
  * Checked **Sequential** (to process files one by one).
  * In the **Items** box, used dynamic content to pull the file list from the previous step:
    ```text
    @activity('Get Metadata1').output.childItems
    ```

#### 3. Step 3: Filter Files by Name (If Condition Activity)
* **Inside ForEach**: Clicked inside the ForEach activity and dragged the **If Condition** activity (`iffilematches`).
* **Expression Configuration**: Used dynamic content to check if each iterated file name starts with `"Fact"`:
  ```text
  @startswith(item().name, 'Fact')
  ```

#### 4. Parameterized Copy Activity (True Activities)
* Inside the **True** activities block, dragged and dropped the **Copy data** activity.
* **Source Configuration**:
  * Set the source to a parameterized source pointing to the folder where all files are present.
  * Created a dataset parameter named `pfile` and set the file path using dynamic content: `pfile`.
  * In the Copy Activity's source settings for parameter `pfile`, added the dynamic value:
    ```text
    @item().name
    ```
* **Sink Configuration**:
  * Pointed the sink to the `reporting` container.
  * Created a dataset parameter named `pfile2` and set the file path in the dataset to `@dataset().pfile2`.
  * In the Copy Activity's sink settings for parameter `pfile2`, added the value:
    ```text
    @item().name
    ```

## ⚙️ How to Add and Configure a Data Flow Activity
1. **Create Pipeline**: Open your pipeline canvas in the Author tab.
2. **Add Activity**: Expand the **Move & transform** category in the activities panel and drag the **Data flow** activity onto your pipeline canvas.
3. **Configure Settings**: 
   * Click on the activity, go to the **Settings** tab, and select your target Data Flow from the dropdown.
   * Choose your Integration Runtime compute properties and configure any parameters if your data flow requires them.
4. **Execute**: Connect it to upstream activities (like a *Get Metadata* or *Wait* activity) or run a **Debug** session to test the Spark-powered transformation live.

## 🌊 Building and Configuring a Mapping Data Flow

#### 1. Create the Data Flow Object
* **Author Hub**: Went to the Author hub, clicked the **+** (plus) icon, and selected **Data flow** -> **Data flow**.
* **Name**: Named the data flow component **`data transformation`**.

#### 2. Configure Data Flow Settings
* **Settings Tab**: Clicked on the settings section and clicked **+ New** under **Data flow**.
* **Name**: Assigned the internal data flow name as **`transformcsv`**.

#### 3. Add and Configure the Source Transformation
* **Add Source**: Clicked on **Add Source** on the data flow canvas.
* **Output Stream Name**: Named the source stream **`sourceCSV`**.
* **Dataset Selection**: 
  * Selected an existing dataset (`dataflow_sorce`).
  * Linked to the correct **linked service** pointing to the storage account.
  * Pointed to the target container/folder (`reporting`) and specified the file format (`csv`).
  * Left file selection as `none` (or general container level) to allow dynamic processing.
* **Source Options**: Configured optional file handling rules (such as wildcard paths or header settings if required).

#### 4. Enable Data Flow Debug & Import Projection
* **Data Flow Debug**: Toggled **Data flow debug** to **ON** at the top of the canvas. 
  * *Explanation*: This spins up an underlying Azure-managed Spark compute cluster (using a configured **Time to Live / TTL**) so you can interactively preview data and test transformations in real-time.
* **Projection Tab**: Clicked **Import projection** once the debug cluster was active.
  * *Explanation*: This instructs Spark to read the source CSV schema, automatically detecting column names and data types (string, integer, date, etc.).

#### 5. Data Preview Tab
* **Data Preview**: Clicked on the **Data preview** tab and clicked **Refresh**.
* *Explanation*: This executes a live query against the debug cluster to fetch and display a tabular sample of your raw CSV data, allowing you to visually verify that columns and rows are being read correctly before applying any transformations.

## 📋Creating and Managing Triggers in ADF

#### 1. Add and Create a New Trigger
1. **Navigate to Pipeline**: Open your completed pipeline in the Author tab.
2. **Add Trigger Menu**: Click on the **Add trigger** dropdown button at the top menu bar of the canvas.
3. **New Trigger**: Select **+ New** from the dropdown menu to open the trigger creation configuration pane.

#### 2. Configure Trigger Properties
* **Name**: Entered the trigger name (e.g., **`selectedfiletrigger`**).
* **Type**: Selected the desired execution type from the options:
  * **Schedule**: Runs pipelines on a recurring time-based calendar schedule (e.g., daily, hourly).
  * **Tumbling Window**: Runs pipelines in a continuous, non-overlapping time sequence (great for sequential data loads).
  * **Storage Event**: Triggers a pipeline automatically whenever a file is created, updated, or deleted in an Azure storage container (e.g., when a new file lands in your raw container).
  * **Custom Events**: Triggers pipelines based on Azure Event Grid custom event subscriptions.
* **Start Date & End Date**: Configured the active date range and time zone for when the trigger should start firing and when it should expire.

#### 3. Publish All Changes
* **Publishing**: Because triggers run at the backend service level, newly created or modified triggers require you to click **Publish all** at the top of the ADF studio so Azure registers and activates the schedule.

#### 4. Manage Tab (Monitoring & Controlling Triggers)
* **Accessing Triggers**: Click on the **Manage (⚙️)** hub on the left sidebar and select **Triggers**.
* **Trigger Control**: From this central dashboard, you can view all created triggers, check their status, and **Start** or **Stop** triggers manually as needed.

## 📋 Working with Variables and Set Variable Activity

#### Step 1: Declare a Pipeline Variable
Before you can use a Set Variable activity, you must create the variable at the pipeline level:
1. Click on the blank background space of your pipeline canvas (ensuring no individual activity is selected).
2. Go to the **Variables** tab at the bottom panel.
3. Click **+ New** and define your variable:
   * **Name**: Give it a clear name (e.g., `vFileStatus` or `vCounter`).
   * **Type**: Choose the data type (`String`, `Bool`, or `Array`).
   * **Value** (Optional): Provide an initial default value.

#### Step 2: Add and Configure the Set Variable Activity
1. **Add Activity**: Expand the **General** category in the activities panel and drag **Set variable** onto your pipeline canvas.
2. **Settings Tab Configuration**:
   * **Name**: Give the activity a descriptive name.
   * **Type Selector (`Pipeline variable` vs `Pipeline return value`)**: 
     * **Pipeline variable**: Select this when updating a custom mutable variable you declared at the pipeline level (like `vFileStatus`). Once selected, choose your variable name from the dropdown.
     * **Pipeline return value**: Select this when you want the pipeline to return a specific output result back to a parent pipeline or triggering process (using the `setPayload` pattern to return execution summaries or status outputs).
   * **Value**: In the value box (or using **Add dynamic content**), specify what you want to assign. 
     * *Example (Static value)*: `Success`
     * *Example (Dynamic value from a previous activity)*: `@activity('GetMetadata1').output.exists`
3. **Execution**: When the pipeline runs and hits this activity, it evaluates the expression and updates the value for any subsequent activities downstream.


## Configuring the Delete Activity

#### Step 1: Add the Activity
1. Open your pipeline in the Author tab.
2. Expand the **Move & transform** (or General/Utilities depending on your UI version) category in the activities panel.
3. Drag the **Delete** activity onto your pipeline canvas.

#### Step 2: Configure Source Dataset & Path Settings
* **Dataset**: Go to the **Source** tab and select the dataset pointing to the container, folder, or file you want to delete. If your dataset uses parameters (like `pfile`), you can pass the dynamic file name or folder path here.
* **Dataset Parameters**: If your delete dataset is parameterized, specify the values (e.g., passing `@item().name` or a specific file path).

#### Step 3: Configure Deletion Rules (Settings Tab)
* **Log settings**: (Optional) Enable logging to record which files were successfully deleted.
* **Recursive**: Check this box if you want to delete an entire folder structure and all its contents recursively. Leave unchecked if you are targeting a single specific file.

#### Step 4: Wire Up Activity Dependencies
* Connect your **Copy Activity** or **Data Flow** to the **Delete Activity** using the standard **On Success (Green Arrow)** connector. This guarantees the file is only deleted if the data migration completed without errors.


## Configuring the Execute Pipeline Activity

#### Step 1: Add the Activity
1. Open your parent pipeline in the Author tab.
2. Expand the **General** (or Iteration & conditionals) category in the activities panel.
3. Drag the **Execute pipeline** activity onto your canvas.

#### Step 2: Configure Settings Tab
* **Name**: Give your activity a descriptive name (e.g., `Exec_Transform_Sales`).
* **Invoked pipeline**: Click the dropdown menu and select the existing child pipeline you want to trigger (e.g., your `data transformation` pipeline).
* **Wait on completion**: Check this box (default) if you want the parent pipeline to wait for the child pipeline to finish and report its success/failure status before proceeding.

#### Step 3: Pass Parameters to the Child Pipeline (Optional)
If your child pipeline expects parameters (like a file name or execution date):
1. Once you select an **Invoked pipeline** that has parameters defined, a **Parameters** section automatically appears below in the settings tab.
2. Click **+ New** or view the parameter list, and assign values using static text or **Add dynamic content** from the parent pipeline (e.g., passing `@pipeline().parameters.BatchID`).

#### Step 4: Accessing Child Outputs
* If you need to reference outputs from the executed child pipeline in subsequent steps, you can use expressions pointing to the execution activity (e.g., `@activity('Exec_Transform_Sales').output.pipelineRunId`).

## 📋 Creating and Configuring a Storage Events Trigger

#### Step 1: Open Trigger Creation
1. Open your completed pipeline in the Author tab.
2. Click on the **Add trigger** dropdown button at the top menu bar of the canvas.
3. Select **+ New** to open the configuration pane.

#### Step 2: Configure Trigger Properties & Type
* **Name**: Enter a descriptive name for your trigger (e.g., `trg_BlobCreated_IngestSales`).
* **Type**: Select **Storage Events** from the dropdown menu.
    * **Schedule**: A time-based calendar trigger that runs your pipeline on a recurring schedule.
    * **Tumbling Window**: A specialized time-based trigger that executes pipelines in a continuous, non-overlapping time sequence.
    * **Storage Event**: An event-driven trigger powered by Azure Event Grid that listens for actions inside an Azure Storage account or ADLS Gen2 container.
    * **Custom Events**: An event-driven trigger that responds to custom topics published via Azure Event Grid.
* **Azure Subscription**: Select the Azure subscription where your storage account resides.
* **Storage account name**: Choose the specific Storage Account you want ADF to monitor.

#### Step 3: Define Blob Filtering Rules
To prevent your pipeline from firing on *every single file change* in the storage account, you configure container and naming filters:
* **Container name**: Specify the exact container name to monitor (e.g., `raw-data` or `landing`).
* **Blob path begins with**: (Optional) Restrict triggers to a specific subfolder path (e.g., `sales/2026/`).
* **Blob path ends with**: (Optional) Filter by file extension or naming pattern to ensure the pipeline only runs for specific file types (e.g., `.csv` or `Fact_`).

#### Step 4: Select Event Triggers
* Choose **Blob created** (fires when a new file is uploaded) or **Blob deleted** (fires when a file is removed). Most ingestion pipelines use **Blob created**.

#### Step 5: Capture Trigger Metadata in Pipeline Parameters (Crucial Step)
When a storage event trigger fires, it automatically passes system variables about the file that just landed into your pipeline. You can capture these using system variables:
* `@triggerBody().folderPath`: Returns the container and folder path where the file landed.
* `@triggerBody().fileName`: Returns the exact name of the file that triggered the run.
* *Tip*: Pass these trigger properties into your pipeline parameters so downstream activities (like a Copy Activity or Get Metadata) can process the exact file dynamically without hardcoding names.

#### Step 6: Publish All
* Click **Publish all** at the top of the ADF studio so Azure registers the Event Grid subscription on your storage account and activates the trigger.

## 🌐 Creating and Configuring a Web Activity

### Step 1: Add the Activity
1. Open your pipeline in the Author tab.
2. Expand the **General** category in the activities panel.
3. Drag the **Web** activity onto your pipeline canvas.
4. (Optional) In the **General** tab, give the activity a descriptive name (e.g., `Call_GitHub_API` or `Trigger_Logic_App`).

### Step 2: Configure the Target Endpoint (Settings Tab)
1. Click on the **Settings** tab.
2. **URL**: Specify the target REST API endpoint. You can hardcode this or click *Add dynamic content* to pass it as a parameter (e.g., `https://raw.githubusercontent.com/...` or `@pipeline().parameters.ApiUrl`).
3. **Method**: Select the appropriate HTTP method from the dropdown:
   * **GET**: To retrieve data from the endpoint.
   * **POST**: To send data to the endpoint (e.g., triggering an alert).
   * **PUT**: To update existing data.
   * **DELETE**: To remove data.

### Step 3: Configure Body and Headers (If Required)
* **Headers**: Click **+ New** to pass specific HTTP headers. Common headers include:
  * Name: `Content-Type`, Value: `application/json`
  * Name: `Authorization`, Value: `Bearer <your_token>`
* **Body**: If you selected **POST** or **PUT**, a Body text box will appear. Enter the JSON payload here. You can use dynamic content to pass variables (e.g., sending the pipeline failure message dynamically to a Logic App).

### Step 4: Configure Authentication
* Choose the **Authentication** method required by the destination API. 
* Options include **None** (for public APIs), **Basic**, **Client Certificate**, or **System Assigned Managed Identity** (best practice when communicating securely with other internal Azure resources).

### Step 5: Handling the Output
When the Web Activity succeeds, it returns the API's response body. You can reference this JSON output in subsequent downstream activities (like a *Set Variable* or *If Condition* activity) using an expression like:
`@activity('YourWebActivityName').output`